# Scores for the penalty-based QAOA notebooks

This notebook computes the leaderboard measures from `solutions/README.md` for two team notebooks in the repository root:

- `Eli_solution.ipynb`: standard QAOA with 2 layers, angles from COBYLA.
- `Artem_solution_based_on_Eli_solution_iqm.ipynb`: the same model with 1 layer, angles from a grid search, run in simulation and on the IQM Emerald quantum computer.

Both solve 5 locations with 16 qubits: one qubit per "location i is visited at position p". Invalid bitstrings are made expensive with a penalty of 100. Both use the real one-way distances, not the symmetric distances of the other leaderboard entries.

The circuits are re-simulated here from the angles printed in those notebooks. The cost only contains Z terms, so every bitstring simply has an energy, and one layer is a phase per bitstring followed by an X rotation on every qubit. The hardware result cannot be recomputed, so its counts are copied from the notebook output.

In [1]:
import itertools
import json
import math
from datetime import datetime
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

N_LOCATIONS = 5
PENALTY = 100
M = N_LOCATIONS - 1
N_QUBITS = M * M

DATA_FILE = next(path for path in (Path("tallinn_osrm_matrix.json"), Path("../tallinn_osrm_matrix.json"), Path("../../tallinn_osrm_matrix.json")) if path.exists())
ROAD_KM = np.array(json.loads(DATA_FILE.read_text(encoding="utf-8"))["distances"], dtype=float)[:N_LOCATIONS, :N_LOCATIONS] / 1000

# x[i, p] is 1 when location i + 1 is visited at position p. It is qubit i * M + p, as in the two notebooks.
index = np.arange(2 ** N_QUBITS)
x = np.array([[(index >> (i * M + p)) & 1 for p in range(M)] for i in range(M)])

distance = sum(ROAD_KM[0, i + 1] * x[i, 0] + ROAD_KM[i + 1, 0] * x[i, M - 1] for i in range(M))
for p in range(M - 1):
    for i in range(M):
        for j in range(M):
            if i != j:
                distance = distance + ROAD_KM[i + 1, j + 1] * x[i, p] * x[j, p + 1]
broken_rules = ((x.sum(axis=1) - 1) ** 2).sum(axis=0) + ((x.sum(axis=0) - 1) ** 2).sum(axis=0)
ENERGY = (distance + PENALTY * broken_rules) / PENALTY   # the scaled Hamiltonian, up to a constant

VALID = broken_rules == 0
SHORTEST = distance[VALID].min()
OPTIMAL = VALID & (distance <= SHORTEST + 1e-9)
print(f"{VALID.sum()} valid tours among {len(index):,} bitstrings, {OPTIMAL.sum()} shortest: {SHORTEST:.3f} km")


def qaoa_probabilities(betas, gammas):
    """Probability of every bitstring after standard QAOA (X mixer), as built by Qiskit's QAOAAnsatz."""
    state = np.full(2 ** N_QUBITS, 2 ** (-N_QUBITS / 2), dtype=complex)
    for beta, gamma in zip(betas, gammas):
        state = (state * np.exp(-1j * gamma * ENERGY)).reshape((2,) * N_QUBITS)
        for axis in range(N_QUBITS):
            state = math.cos(beta) * state - 1j * math.sin(beta) * np.flip(state, axis=axis)
        state = state.reshape(-1)
    return np.abs(state) ** 2


def shots_for_99(p_opt):
    return math.inf if p_opt <= 0 else math.ceil(math.log(0.01) / math.log1p(-p_opt))


def measures(probabilities, layers):
    p_valid, p_opt = probabilities[VALID].sum(), probabilities[OPTIMAL].sum()
    shots = shots_for_99(p_opt)
    return {"P(valid)": p_valid, "P(opt)": p_opt, "mean gap": probabilities[VALID] @ distance[VALID] / p_valid / SHORTEST - 1,
            "shots": shots, "evaluations": shots * (layers + 1)}

24 valid tours among 65,536 bitstrings, 1 shortest: 16.965 km


# Eli's notebook: 2 layers

Angles from the output of the "QAOA optimization" cell at commit `e934c6b`. That notebook reports P(valid) = 7.5965 % and P(opt) = 0.3249 %, which this simulation must reproduce.

In [2]:
eli = measures(qaoa_probabilities(betas=[-0.49611165, 2.95888332], gammas=[0.16861332, 0.5228545]), layers=2)
print(f"P(valid) = {eli['P(valid)']:.4%}   P(opt) = {eli['P(opt)']:.4%}   mean gap = {eli['mean gap']:.1%}")

P(valid) = 7.5965%   P(opt) = 0.3249%   mean gap = 26.2%


# Artem's notebook: 1 layer, simulated

Angles from the output of the grid-search cell at commit `ad8f8fc`. That notebook reports P(valid) = 5.66 %. It maximises P(valid) and then picks the shortest valid tour from the samples.

In [3]:
artem = measures(qaoa_probabilities(betas=[0.4527], gammas=[5.8060]), layers=1)
print(f"P(valid) = {artem['P(valid)']:.2%}   P(opt) = {artem['P(opt)']:.4%}   mean gap = {artem['mean gap']:.1%}")

P(valid) = 5.66%   P(opt) = 0.1988%   mean gap = 28.5%


# Artem's notebook: 1 layer, on IQM Emerald

The same circuit ran on the 54-qubit IQM Emerald device with 5,000 shots. The notebook output at commit `ad8f8fc` reports 0.1 % valid shots, no shortest tour, and 19.831 km as the best sampled tour. The raw counts are not stored, so the mean gap cannot be computed.

In [4]:
HARDWARE_SHOTS, HARDWARE_VALID, HARDWARE_OPTIMAL, HARDWARE_BEST_KM = 5000, 5, 0, 19.831
hardware = {"P(valid)": HARDWARE_VALID / HARDWARE_SHOTS, "P(opt)": HARDWARE_OPTIMAL / HARDWARE_SHOTS}
print(f"P(valid) = {hardware['P(valid)']:.2%}   P(opt) = {hardware['P(opt)']:.2%}   "
      f"best sampled tour is {HARDWARE_BEST_KM / SHORTEST - 1:.1%} longer than the shortest")

P(valid) = 0.10%   P(opt) = 0.00%   best sampled tour is 16.9% longer than the shortest


# Leaderboard rows

Random guessing here means a uniformly random 16-bit string, which is what these circuits start from.

In [5]:
uniform = measures(np.full(2 ** N_QUBITS, 2.0 ** -N_QUBITS), layers=0)


def pct(value):
    return f"{100 * value:.1f} %" if value >= 0.001 else f"{100 * value:.2g} %"


rows = [[name, layers, pct(m["P(opt)"]), pct(m["P(valid)"]), pct(m["mean gap"]), f"{m['shots']:,}", f"{m['evaluations']:,}"]
        for name, layers, m in (("Eli, simulated", 2, eli), ("Artem, simulated", 1, artem), ("Random 16-bit string", 0, uniform))]
rows.insert(2, ["Artem, IQM Emerald", 1, pct(hardware["P(opt)"]) if hardware["P(opt)"] else "0 of 5,000 shots", pct(hardware["P(valid)"]),
                "not available", "not reached", "not reached"])
headers = ["Solution", "Layers", "P(opt)", "P(valid)", "Mean gap", "Shots for 99 %", "Cost evaluations for 99 %"]
display(Markdown("\n".join(["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
                           + ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows])))

zz_terms = sum(1 for a in range(N_QUBITS) for b in range(a + 1, N_QUBITS)
               if a // M == b // M or a % M == b % M or (abs(a % M - b % M) == 1 and a // M != b // M))
print(f"{zz_terms} ZZ terms per layer, so {2 * zz_terms} CX gates per layer before routing")
print(f"Evaluated {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}")

| Solution | Layers | P(opt) | P(valid) | Mean gap | Shots for 99 % | Cost evaluations for 99 % |
|---|---|---|---|---|---|---|
| Eli, simulated | 2 | 0.3 % | 7.6 % | 26.2 % | 1,416 | 4,248 |
| Artem, simulated | 1 | 0.2 % | 5.7 % | 28.5 % | 2,315 | 4,630 |
| Artem, IQM Emerald | 1 | 0 of 5,000 shots | 0.1 % | not available | not reached | not reached |
| Random 16-bit string | 0 | 0.0015 % | 0.037 % | 26.5 % | 301,803 | 301,803 |

84 ZZ terms per layer, so 168 CX gates per layer before routing
Evaluated 2026-11-12 21:19 EET
